In [ ]:
# ==============================================================================
# Rubric-Count Ablation Experiment
#
# Independent variable : number of rubric criteria N ∈ {5,6,7,8,9,10,"unspecified"}
# Controlled split     : train = rows 0–19, test = rows 20–39 (fixed for all N)
# Output               : ./results_rubric_experiment/<annotator_id>/<N>.json
#                        ./results_rubric_experiment/summary.json
# ==============================================================================
import os
import json
import base64
import time
import random
import re
import urllib.request
import urllib.error
import http.client
import concurrent.futures
from typing import List, Dict, Any, Optional
from pathlib import Path
import pandas as pd
from PIL import Image
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# ==============================================================================
# CONFIGURATION
# ==============================================================================
ANNOTATOR_DIR = "./annotator"
IMAGES_DIR    = "./screen"
RESULTS_DIR   = "./results_rubric_experiment"
MODEL         = "gpt-5.4"

TRAIN_ROWS    = 30          # first N rows → train
TEST_ROWS     = 30          # next  N rows → test (rows 20–39)
SEED          = 7

# The rubric counts to sweep. "unspecified" tells the model to choose freely.
RUBRIC_COUNTS = [5, 6, 7, 8, 9, 10, "unspecified"]

MAX_WORKERS              = 7
PHASE_D_VOTE_RUNS        = 3
NON_DISCRIMINATING_MARGIN = 1.0
TIE_MARGIN               = 2.0

AZURE_OPENAI_TARGET_URI = os.getenv(
    "AZURE_OPENAI_TARGET_URI",
    f"https://wu-lab-east-us-2.openai.azure.com/openai/deployments/{MODEL}/chat/completions?api-version=2025-01-01-preview"
)
AZURE_OPENAI_API_KEY = "XXX"

# ==============================================================================
# HELPERS — identical to original
# ==============================================================================
def set_seed(seed: int):
    random.seed(seed)

MAX_SIZE = (1024, 1024)

def encode_image(image_path: str) -> Optional[str]:
    try:
        img = Image.open(image_path).convert("RGB")
        img.thumbnail(MAX_SIZE, Image.Resampling.LANCZOS)
        import io
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=85)
        return base64.b64encode(buf.getvalue()).decode("utf-8").replace("\n", "")
    except Exception as e:
        print(f"  [!] Error encoding image {image_path}: {e}")
        return None

def extract_json_from_text(text: str) -> dict:
    try:
        match = re.search(r'```(?:json)?\s*(.*?)\s*```', text, re.DOTALL)
        if match:
            return json.loads(match.group(1))
        return json.loads(text)
    except json.JSONDecodeError as e:
        print(f"  [!] JSON Parse Error. Raw text:\n{text}\n")
        raise e

def is_valid_phase_a_result(result: Any) -> bool:
    return isinstance(result, dict) and isinstance(result.get("preferred_features"), list)

def is_valid_phase_d_result(result: Any) -> bool:
    return isinstance(result, dict) and isinstance(result.get("criteria_evaluations"), list)

def call_llm(
    prompt: str,
    image_a_b64: Optional[str] = None,
    image_b_b64: Optional[str] = None,
    max_retries: int = 5,
    temperature: float = 0.2,
) -> Dict[str, Any]:
    content = [{"type": "text", "text": prompt}]
    if image_a_b64 and image_b_b64:
        content.extend([
            {"type": "text",      "text": "Image A:"},
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_a_b64}"}},
            {"type": "text",      "text": "Image B:"},
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_b_b64}"}},
        ])
    payload = {
        "messages": [{"role": "user", "content": content}],
        "temperature": temperature,
        "max_completion_tokens": 1200,
    }
    headers = {
        "Content-Type": "application/json",
        "api-key": AZURE_OPENAI_API_KEY,
        "User-Agent": "Jupyter-LLM-Evaluator/1.0",
    }
    data = json.dumps(payload).encode("utf-8")
    for attempt in range(max_retries):
        try:
            req = urllib.request.Request(
                AZURE_OPENAI_TARGET_URI, data=data, headers=headers, method="POST"
            )
            with urllib.request.urlopen(req, timeout=30) as response:
                result = json.loads(response.read().decode("utf-8"))
                return extract_json_from_text(result["choices"][0]["message"]["content"])
        except urllib.error.HTTPError as e:
            body = ""
            try:
                body = e.read().decode("utf-8")
            except Exception:
                pass
            print(f"  [!] HTTP {e.code} (Attempt {attempt+1}/{max_retries}): {body or e.reason}")
            if e.code == 400:
                raise
            if attempt < max_retries - 1:
                time.sleep(2 + (2 ** attempt))
            else:
                raise
        except (urllib.error.URLError, http.client.RemoteDisconnected, ConnectionResetError) as e:
            print(f"  [!] Connection Error (Attempt {attempt+1}/{max_retries}): {e}")
            if attempt < max_retries - 1:
                time.sleep(2 + (2 ** attempt))
            else:
                raise
        except Exception as e:
            print(f"  [!] API Error (Attempt {attempt+1}/{max_retries}): {e}")
            if attempt < max_retries - 1:
                time.sleep(2 + (2 ** attempt))
            else:
                raise
    return {}

# ==============================================================================
# DATA HELPERS
# ==============================================================================
def get_winner_label(row: pd.Series) -> str:
    choice = str(row.get("final_choice", "")).strip()
    return "B" if choice in ("A < B", "A << B") else "A"

def get_preference_strength(row: pd.Series) -> int:
    choice = str(row.get("final_choice", "")).strip()
    return 2 if (">>" in choice or "<<" in choice) else 1

def load_annotator_data(csv_path: str) -> pd.DataFrame:
    return pd.read_csv(csv_path)

def fixed_split(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Train = first TRAIN_ROWS rows.
    Test  = next  TEST_ROWS rows (rows TRAIN_ROWS .. TRAIN_ROWS+TEST_ROWS-1).
    Raises if the CSV doesn't have enough rows.
    """
    required = TRAIN_ROWS + TEST_ROWS
    if len(df) < required:
        raise ValueError(
            f"CSV has only {len(df)} rows but experiment needs {required} "
            f"({TRAIN_ROWS} train + {TEST_ROWS} test)."
        )
    train = df.iloc[:TRAIN_ROWS].reset_index(drop=True)
    test  = df.iloc[TRAIN_ROWS : TRAIN_ROWS + TEST_ROWS].reset_index(drop=True)
    return train, test

# ==============================================================================
# CHECKPOINT HELPERS
# ==============================================================================
def phase_a_cache_path(annotator_id: str, rubric_count) -> str:
    return os.path.join(RESULTS_DIR, annotator_id, f"phase_a_{rubric_count}.json")

def condition_result_path(annotator_id: str, rubric_count) -> str:
    return os.path.join(RESULTS_DIR, annotator_id, f"{rubric_count}.json")

def partial_phase_d_path(annotator_id: str, rubric_count) -> str:
    return os.path.join(RESULTS_DIR, annotator_id, f"{rubric_count}_partial_d.json")

def load_json(path: str) -> Any:
    with open(path) as f:
        return json.load(f)

def save_json(path: str, obj: Any):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

# ==============================================================================
# PHASE A — unchanged from original
# ==============================================================================
def phase_a(train_df: pd.DataFrame, annotator_id: str, rubric_count) -> list:
    if rubric_count == "unspecified":
        dimension_instruction = "Identify as many design dimensions as you observe differences in — no fixed limit."
    else:
        dimension_instruction = f"Identify up to {rubric_count} design dimensions where they differ."

    print(f"\n  --- Phase A: Pair Analysis ({len(train_df)} pairs / {MAX_WORKERS} workers / rubric_count={rubric_count}) ---")
    analysis_results = []

    def _worker(idx, row):
        cid        = row.get("cid", "unknown")
        img_a_b64  = encode_image(os.path.join(IMAGES_DIR, row["left_file"]))
        img_b_b64  = encode_image(os.path.join(IMAGES_DIR, row["right_file"]))
        if not img_a_b64 or not img_b_b64:
            return {"skip": True, "reason": "missing image"}

        winner              = get_winner_label(row)
        loser               = "B" if winner == "A" else "A"
        preference_strength = get_preference_strength(row)
        final_choice_raw    = str(row.get("final_choice", "")).strip()
        intensity_note = (
            "This was a STRONG preference (the user was very decisive)."
            if preference_strength == 2
            else "This was a MILD preference (the user had a slight lean)."
        )

        ANALYSIS_PROMPT = f"""You are an expert aesthetic preference researcher uncovering a user's NICHE and INDIVIDUAL visual taste from UI choices. Prioritize traits that reveal distinctive personal aesthetic sensitivities, even if they are unconventional or not objectively better UX. Focus on what the choice implies about the user’s unique stylistic preferences rather than general design best practices.
The user looked at two UI designs: Design A and Design B.
The user's raw choice label was: "{final_choice_raw}"
The user EXPLICITLY CHOSE: Design {winner}. {intensity_note}

Look at both designs carefully.

1. {dimension_instruction}
2. For each dimension, state the trait in the CHOSEN design ({winner}) vs the REJECTED design ({loser}).
3. Also write a "why_user_rejected_loser" summary — what specifically made the rejected design not chosen.
4. For each feature, assign a confidence level: "high" if the difference is visually obvious and clearly aligns with the user's choice, "medium" if plausible, "low" if speculative.

Return ONLY a JSON object with this strict schema:
{{
  "why_user_chose_winner": "<max 15 words — specific visual reason>",
  "why_user_rejected_loser": "<max 15 words — what made Design {loser} unacceptable>",
  "preferred_features": [
    {{
      "dimension": "<snake_case_name>",
      "preferred_trait_in_winner": "<brief trait in Design {winner}>",
      "rejected_trait_in_loser": "<brief trait in Design {loser}>",
      "confidence": "<high|medium|low>"
    }}
  ]
}}"""

        try:
            result = call_llm(ANALYSIS_PROMPT, img_a_b64, img_b_b64)
            return {
                "skip": False,
                "pair_id": row.get("pair_id"),
                "cid": cid,
                "human_choice": winner,
                "preference_strength": preference_strength,
                "final_choice_raw": final_choice_raw,
                "llm_output_valid": is_valid_phase_a_result(result),
                "llm_analysis": result,
            }
        except Exception as e:
            return {
                "skip": False,
                "pair_id": row.get("pair_id"),
                "cid": cid,
                "human_choice": winner,
                "preference_strength": preference_strength,
                "final_choice_raw": final_choice_raw,
                "llm_output_valid": False,
                "llm_analysis": None,
                "error": str(e),
            }

    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(_worker, idx, row): (idx, row) for idx, row in train_df.iterrows()}
        for future in concurrent.futures.as_completed(futures):
            idx, row = futures[future]
            cid = row.get("cid", "unknown")
            try:
                res = future.result()
            except Exception as e:
                print(f"  -> Fatal Thread Error on row {idx}: {e}")
                continue
            if res.get("skip"):
                print(f"  -> Skipping row {idx} ({res.get('reason')})")
                continue
            analysis_results.append({k: v for k, v in res.items() if k != "skip"})
            if "error" in res:
                print(f"  Analyzed [{cid}] -> Failed: {res.get('error')}")
            else:
                print(
                    f"  Analyzed [{cid}] [choice={res.get('final_choice_raw')}] "
                    f"[winner={res.get('human_choice')}] [str={res.get('preference_strength')}] "
                    f"-> Valid: {res.get('llm_output_valid')}"
                )
    return analysis_results

# ==============================================================================
# PHASE C — parameterised by rubric_count
# ==============================================================================
def phase_c(analysis_results: list, rubric_count) -> tuple[str, list]:
    """
    rubric_count : int (5–10) or the string "unspecified"
    """
    # Build the count instruction and schema key dynamically
    if rubric_count == "unspecified":
        count_instruction = (
            "Identify as many evaluation criteria as you think are well-supported by the data. "
            "There is no fixed number — quality matters more than quantity."
        )
        schema_key        = "evaluation_criteria"   # model may return any number
        top_n_label       = "evaluation_criteria"
    else:
        count_instruction = f"Identify exactly the TOP {rubric_count} evaluation criteria."
        schema_key        = f"top_{rubric_count}_evaluation_criteria"
        top_n_label       = schema_key

    print(f"\n  --- Phase C: Synthesis (rubric_count={rubric_count}, {len(analysis_results)} analyses) ---")

    observations = []
    for a in analysis_results:
        strength = int(a.get("preference_strength", 1) or 1)
        raw_features = (
            a.get("llm_analysis", {}).get("preferred_features", [])
            if a.get("llm_analysis") else []
        )
        filtered = [f for f in raw_features if isinstance(f, dict) and f.get("confidence") != "low"]
        observations.append({
            "final_choice_raw":    a.get("final_choice_raw"),
            "preference_strength": strength,
            "preferred_features":  [{**f, "evidence_weight": strength} for f in filtered],
        })

    observations.sort(key=lambda x: x["preference_strength"], reverse=True)
    sample_observations = observations[:50]

    SYNTHESIS_PROMPT = f"""You are a UX researcher. You are given raw observations of what a specific user PREFERRED vs REJECTED across multiple UI pairs.

Raw Observations (sorted strongest preference first):
{json.dumps(sample_observations, indent=2)}

Identify the strongest, most consistent patterns in what they PREFER.

Each observation includes `preference_strength` and each feature includes `evidence_weight`.
- weight 1 = weak preference
- weight 2 = strong preference

CRITICAL INSTRUCTIONS:
1. {count_instruction}
   Group similar concepts, sum their weighted evidence, and assign a `weight` to each.
2. Strong-preference evidence (weight 2) must count MORE than weak-preference evidence (weight 1).
3. MINIMUM EVIDENCE: only include a criterion if it appears in AT LEAST 2 separate pairs.
4. Identify dimensions where the user was CONTRADICTORY and list them in "contradictory_signals".
   Do NOT include those in the returned criteria.

Return a JSON object with this schema:
{{
  "niche_preference_profile": "<Clear, literal description of the visual traits this user likes>",
  "contradictory_signals": ["<dimension_name>", ...],
  "{top_n_label}": [
    {{
      "criterion":    "<snake_case_criterion>",
      "description":  "<exact visual trait the user prefers>",
      "weight":       <numeric_float>
    }}
  ]
}}
"""

    print("  Synthesizing preference profile...")
    try:
        result = call_llm(prompt=SYNTHESIS_PROMPT)
        niche_preference_profile = result.get("niche_preference_profile", "Could not synthesize profile.")
        criteria_raw = result.get(top_n_label, [])
        contradictory = result.get("contradictory_signals", [])
        if contradictory:
            print(f"  Contradictory dimensions excluded: {contradictory}")
        if not criteria_raw:
            # fallback: try any key that looks like a list of criteria
            for v in result.values():
                if isinstance(v, list) and v and isinstance(v[0], dict) and "criterion" in v[0]:
                    criteria_raw = v
                    break
        if not criteria_raw:
            raise ValueError("No criteria extracted")
    except Exception as e:
        print(f"  -> Synthesis failed: {e}")
        niche_preference_profile = "Synthesis failed."
        criteria_raw = []

    clean_criteria = []
    for i, c in enumerate(criteria_raw):
        if isinstance(c, dict):
            clean_criteria.append({
                "criterion":   c.get("criterion", f"criterion_{i+1}"),
                "description": c.get("description", ""),
                "weight":      float(c.get("weight", 1.0) or 1.0),
            })
        else:
            clean_criteria.append({"criterion": f"criterion_{i+1}", "description": str(c), "weight": 1.0})

    if not clean_criteria:
        fallback_n = 7 if rubric_count == "unspecified" else int(rubric_count)
        clean_criteria = [
            {"criterion": f"fallback_{i+1}", "description": "fallback", "weight": 1.0}
            for i in range(fallback_n)
        ]

    print(f"  Profile: {niche_preference_profile[:120]}...")
    for c in clean_criteria:
        print(f"    - {c['criterion']} (weight={c['weight']})")
    return niche_preference_profile, clean_criteria

# ==============================================================================
# PHASE D — unchanged from original, accepts criteria list of any length
# ==============================================================================
def _score_pair_once(
    img_a_b64: str,
    img_b_b64: str,
    niche_preference_profile: str,
    criteria_for_eval: list,
    weights_map: dict,
    temperature: float = 0.3,
) -> dict:
    criteria_json = json.dumps(criteria_for_eval, indent=2)
    EVAL_PROMPT = f"""You are an AI scoring two UI designs for a user with this EXACT preference profile:

USER'S NICHE DESIGN PREFERENCE PROFILE:
"{niche_preference_profile}"

Evaluation criteria (user's preferred trait per dimension):
{criteria_json}

CRITICAL INSTRUCTIONS:
1. For EACH criterion, assign a RELATIVE score: how much better Design A satisfies this criterion
   compared to Design B, on a scale from -5 to +5.
   Positive = A is better | Negative = B is better | 0 = genuinely equal.
2. Use the full range. Do not cluster at 0.
3. Score ONLY against the user's specific preferred trait — NOT generic best practices.
4. Add "confidence": "high" | "medium" | "low".

Return ONLY a JSON object:
{{
  "criteria_evaluations": [
    {{
      "criterion": "<must match one of the provided criteria>",
      "relative_score_a_minus_b": <integer -5 to +5>,
      "reason": "<max 7 words>",
      "confidence": "<high|medium|low>"
    }}
  ]
}}
"""
    result = call_llm(EVAL_PROMPT, img_a_b64, img_b_b64, temperature=temperature)
    if not is_valid_phase_d_result(result):
        return {"valid": False, "total_delta": 0.0, "evaluations": []}

    total_delta = 0.0
    evaluations = []
    for item in result.get("criteria_evaluations", []):
        crit_name  = item.get("criterion", "")
        confidence = item.get("confidence", "medium")
        try:
            delta = float(item.get("relative_score_a_minus_b", 0))
        except (ValueError, TypeError):
            delta = 0.0

        effective_delta = 0.0 if (confidence == "low" or abs(delta) < NON_DISCRIMINATING_MARGIN) else delta
        weight          = weights_map.get(crit_name, 1.0)
        weighted_delta  = effective_delta * weight

        evaluations.append({**item, "effective_delta": effective_delta,
                             "applied_weight": weight, "weighted_delta": weighted_delta})
        total_delta += weighted_delta

    return {"valid": True, "total_delta": total_delta, "evaluations": evaluations}


def phase_d(
    test_df: pd.DataFrame,
    niche_preference_profile: str,
    criteria: list,
    partial_path: str,                    # path for per-pair checkpoint writes
    completed_predictions: list = None,   # already-done results loaded from partial file
) -> tuple[list, float, int, int]:
    completed_predictions = completed_predictions or []
    done_pair_ids = {r["pair_id"] for r in completed_predictions if r.get("pair_id") is not None}

    remaining_df = test_df[~test_df["pair_id"].isin(done_pair_ids)].reset_index(drop=True)
    skipped = len(test_df) - len(remaining_df)
    print(
        f"\n  --- Phase D: Evaluation ({len(test_df)} pairs total | "
        f"{skipped} already done | {len(remaining_df)} remaining | "
        f"{MAX_WORKERS} workers / {PHASE_D_VOTE_RUNS} votes) ---"
    )

    criteria_for_eval = [{"criterion": c["criterion"], "preferred_trait": c["description"]} for c in criteria]
    weights_map       = {c["criterion"]: float(c.get("weight", 1.0)) for c in criteria}

    # Shared mutable state guarded by a lock for the partial-file writer
    import threading
    write_lock          = threading.Lock()
    all_predictions     = list(completed_predictions)   # start with already-done results

    def _flush_partial():
        with write_lock:
            save_json(partial_path, all_predictions)

    def _worker(idx, row):
        img_a_b64 = encode_image(os.path.join(IMAGES_DIR, row["left_file"]))
        img_b_b64 = encode_image(os.path.join(IMAGES_DIR, row["right_file"]))
        if not img_a_b64 or not img_b_b64:
            return {"skip": True, "reason": "missing image"}

        true_winner      = get_winner_label(row)
        final_choice_raw = str(row.get("final_choice", "")).strip()
        true_strength    = get_preference_strength(row)

        try:
            runs        = [_score_pair_once(img_a_b64, img_b_b64, niche_preference_profile,
                                            criteria_for_eval, weights_map, temperature=0.3)
                           for _ in range(PHASE_D_VOTE_RUNS)]
            valid_runs  = [r for r in runs if r["valid"]]
            if not valid_runs:
                raise ValueError("All scoring runs invalid")

            avg_delta = sum(r["total_delta"] for r in valid_runs) / len(valid_runs)
            votes_a   = sum(1 for r in valid_runs if r["total_delta"] > 0)
            votes_b   = sum(1 for r in valid_runs if r["total_delta"] < 0)
            votes_tie = len(valid_runs) - votes_a - votes_b

            predicted = (
                ("A" if votes_a >= votes_b else "B")
                if abs(avg_delta) < TIE_MARGIN
                else ("A" if avg_delta > 0 else "B")
            )

            best_run = max(valid_runs, key=lambda r: abs(r["total_delta"]))
            return {
                "skip": False,
                "pair_id": row.get("pair_id"),
                "cid": row.get("cid", "unknown"),
                "true_winner": true_winner,
                "final_choice_raw": final_choice_raw,
                "true_strength": true_strength,
                "predicted_choice": predicted,
                "is_correct": predicted == true_winner,
                "llm_output_valid": True,
                "details": {
                    "criteria_evaluations": best_run["evaluations"],
                    "avg_delta": avg_delta,
                    "votes_a": votes_a, "votes_b": votes_b, "votes_tie": votes_tie,
                    "derived_predicted_choice": predicted,
                },
            }
        except Exception as e:
            return {
                "skip": False,
                "pair_id": row.get("pair_id"),
                "cid": row.get("cid", "unknown"),
                "true_winner": true_winner,
                "final_choice_raw": final_choice_raw,
                "true_strength": true_strength,
                "predicted_choice": None,
                "is_correct": False,
                "llm_output_valid": False,
                "details": None,
                "error": str(e),
            }

    correct_predictions = sum(1 for r in completed_predictions if r.get("is_correct") and r.get("llm_output_valid"))
    valid_evals         = sum(1 for r in completed_predictions if r.get("llm_output_valid"))

    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(_worker, idx, row): (idx, row) for idx, row in remaining_df.iterrows()}
        for future in concurrent.futures.as_completed(futures):
            idx, row = futures[future]
            cid = row.get("cid", "unknown")
            try:
                res = future.result()
            except Exception as e:
                print(f"  -> Fatal Thread Error on row {idx}: {e}")
                continue
            if res.get("skip"):
                print(f"  -> Skipping row {idx} ({res.get('reason')})")
                continue

            entry = {k: v for k, v in res.items() if k != "skip"}
            if res["llm_output_valid"]:
                valid_evals += 1
                if res["is_correct"]:
                    correct_predictions += 1

            with write_lock:
                all_predictions.append(entry)
            _flush_partial()   # write after every completed pair

            running_acc = (correct_predictions / valid_evals) if valid_evals else 0.0
            details     = res.get("details") or {}
            verdict     = "RIGHT" if res.get("is_correct") else "WRONG"
            print(
                f"  Eval [{cid}] avg_delta={details.get('avg_delta', 0):.2f} | "
                f"votes={details.get('votes_a','-')}A/{details.get('votes_b','-')}B | "
                f"Pred={res.get('predicted_choice')} | True={res.get('true_winner')} | {verdict} | "
                f"Acc={correct_predictions}/{valid_evals} ({running_acc*100:.2f}%)"
            )

    accuracy = (correct_predictions / valid_evals) if valid_evals else 0.0
    print(f"  Final Accuracy: {correct_predictions}/{valid_evals} ({accuracy*100:.2f}%)")
    return all_predictions, accuracy, correct_predictions, valid_evals

# ==============================================================================
# PER-ANNOTATOR, PER-RUBRIC-COUNT RUNNER
# ==============================================================================
def run_condition(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    annotator_id: str,
    rubric_count,
    phase_a_results: list,
) -> dict:
    label        = str(rubric_count)
    out_path     = condition_result_path(annotator_id, rubric_count)
    partial_path = partial_phase_d_path(annotator_id, rubric_count)

    print(f"\n{'─'*55}")
    print(f"  Annotator={annotator_id}  |  rubric_count={label}")
    print(f"{'─'*55}")

    # Phase C
    niche_preference_profile, criteria = phase_c(phase_a_results, rubric_count)

    # Load any partially-completed Phase D results
    completed_predictions = []
    if os.path.exists(partial_path):
        try:
            completed_predictions = load_json(partial_path)
            print(f"  Resuming Phase D: loaded {len(completed_predictions)} completed pair(s) from {partial_path}")
        except Exception as e:
            print(f"  [!] Could not load partial Phase D file ({e}), starting fresh.")

    # Phase D
    eval_predictions, accuracy, correct, total_eval = phase_d(
        test_df, niche_preference_profile, criteria,
        partial_path=partial_path,
        completed_predictions=completed_predictions,
    )

    result = {
        "annotator_id":            annotator_id,
        "rubric_count":            label,
        "actual_criteria_count":   len(criteria),
        "model":                   MODEL,
        "split": {
            "train_rows":      len(train_df),
            "test_rows":       len(test_df),
            "train_start_row": 0,
            "train_end_row":   TRAIN_ROWS - 1,
            "test_start_row":  TRAIN_ROWS,
            "test_end_row":    TRAIN_ROWS + TEST_ROWS - 1,
        },
        "niche_preference_profile": niche_preference_profile,
        "criteria":                 criteria,
        "evaluation_metrics": {
            "accuracy":        accuracy,
            "correct":         correct,
            "total_evaluated": total_eval,
        },
        "predictions": eval_predictions,
    }
    save_json(out_path, result)
    print(f"  Saved → {out_path}")

    # Clean up partial file now that the full result is written
    if os.path.exists(partial_path):
        os.remove(partial_path)
        print(f"  Removed partial file {partial_path}")

    return result


def run_pipeline_for_annotator(csv_path: str, annotator_id: str) -> list[dict]:
    print(f"\n{'='*60}")
    print(f"Annotator: {annotator_id}")
    print(f"{'='*60}")
    set_seed(SEED)

    df = load_annotator_data(csv_path)
    print(f"  Loaded {len(df)} rows")

    train_df, test_df = fixed_split(df)
    print(f"  Train rows: 0–{TRAIN_ROWS-1}  |  Test rows: {TRAIN_ROWS}–{TRAIN_ROWS+TEST_ROWS-1}")

    os.makedirs(os.path.join(RESULTS_DIR, annotator_id), exist_ok=True)

    # ── Sweep rubric counts — Phase A re-runs per condition ──
    condition_results = []
    for rubric_count in RUBRIC_COUNTS:
        label    = str(rubric_count)
        out_path = condition_result_path(annotator_id, rubric_count)

        # Skip entirely if the final condition result already exists
        if os.path.exists(out_path):
            print(f"\n  [SKIP] rubric_count={label} already complete ({out_path})")
            try:
                condition_results.append(load_json(out_path))
            except Exception as e:
                print(f"  [!] Could not reload {out_path}: {e}")
            continue

        # Load Phase A from cache if available, otherwise run and cache it
        cache_path = phase_a_cache_path(annotator_id, rubric_count)
        if os.path.exists(cache_path):
            print(f"\n  [RESUME] Loading cached Phase A for rubric_count={label} from {cache_path}")
            try:
                phase_a_results = load_json(cache_path)
            except Exception as e:
                print(f"  [!] Could not load Phase A cache ({e}), re-running Phase A.")
                phase_a_results = phase_a(train_df, annotator_id, rubric_count)
                save_json(cache_path, phase_a_results)
        else:
            print(f"\n  Running Phase A for rubric_count={label} ...")
            phase_a_results = phase_a(train_df, annotator_id, rubric_count)
            save_json(cache_path, phase_a_results)
            print(f"  Phase A saved → {cache_path}")

        result = run_condition(train_df, test_df, annotator_id, rubric_count, phase_a_results)
        condition_results.append(result)

    return condition_results


# ==============================================================================
# MAIN
# ==============================================================================
def main():
    annotator_dir = Path(ANNOTATOR_DIR)
    if not annotator_dir.exists():
        print(f"Error: {ANNOTATOR_DIR} directory not found.")
        return
    csv_files = sorted(annotator_dir.glob("*.csv"))
    if not csv_files:
        print(f"No CSV files found in {ANNOTATOR_DIR}")
        return

    print(f"Found {len(csv_files)} annotator(s): {[f.stem for f in csv_files]}")
    print(f"Rubric counts to sweep: {RUBRIC_COUNTS}")
    print(f"Fixed split: train=rows 0–{TRAIN_ROWS-1}, test=rows {TRAIN_ROWS}–{TRAIN_ROWS+TEST_ROWS-1}")

    os.makedirs(RESULTS_DIR, exist_ok=True)

    all_results: list[dict] = []
    for csv_path in csv_files:
        annotator_id = csv_path.stem
        try:
            results = run_pipeline_for_annotator(str(csv_path), annotator_id)
            all_results.extend(results)
        except Exception as e:
            print(f"\n[ERROR] Failed for annotator {annotator_id}: {e}")

    # ── Summary table ──
    summary_rows = []
    for r in all_results:
        metrics = r.get("evaluation_metrics", {})
        summary_rows.append({
            "annotator_id":          r["annotator_id"],
            "rubric_count":          r["rubric_count"],
            "actual_criteria_count": r["actual_criteria_count"],
            "accuracy":              metrics.get("accuracy"),
            "correct":               metrics.get("correct"),
            "total_evaluated":       metrics.get("total_evaluated"),
        })

    summary = {
        "experiment": "rubric_count_ablation",
        "rubric_counts_tested": [str(x) for x in RUBRIC_COUNTS],
        "fixed_split": {"train": TRAIN_ROWS, "test": TEST_ROWS},
        "results": summary_rows,
    }
    summary_path = os.path.join(RESULTS_DIR, "summary.json")
    with open(summary_path, "w") as f:
        json.dump(summary, f, indent=2)

    # Pretty-print summary table
    print(f"\n{'='*60}")
    print("EXPERIMENT SUMMARY")
    print(f"{'='*60}")
    print(f"{'Annotator':<20} {'RubricCount':<14} {'ActualN':<10} {'Acc':>8}  {'Correct':>8}")
    print("-" * 62)
    for row in summary_rows:
        acc = row["accuracy"]
        acc_str = f"{acc*100:.1f}%" if acc is not None else "N/A"
        print(
            f"{row['annotator_id']:<20} {row['rubric_count']:<14} "
            f"{row['actual_criteria_count']:<10} {acc_str:>8}  "
            f"{row['correct']}/{row['total_evaluated']}"
        )
    print(f"\nFull summary → {summary_path}")
    print("Done.")


if __name__ == "__main__":
    main()

Found 20 annotator(s): ['0c65ba0b46894372', '1d3ee9b46ac34e6c', '247b7dfa8a5347ad', '2c79548f2ab44243', '2d8219ac74d146ba', '454c297fa1e54296', '498b9ea72d994e8e', '653f05d5c9ab4c97', '6ccbe484cb96425f', '7602a5d37b7d4220', '7a945bb87f2b4cd2', '8287fcc5b6504e39', '8f61f5a0685f42ec', '97945b44a2b54914', 'a692cdb11bbe432c', 'abda3f52c24c4038', 'ad7def63e86045b1', 'dad4b876ad3147e4', 'e8f37a526c444958', 'eee1fd24aad648ed']
Rubric counts to sweep: [5, 6, 7, 8, 9, 10, 'unspecified']
Fixed split: train=rows 0–29, test=rows 30–59

Annotator: 0c65ba0b46894372
  Loaded 610 rows
  Train rows: 0–29  |  Test rows: 30–59

  [SKIP] rubric_count=5 already complete (./results_rubric_experiment/0c65ba0b46894372/5.json)

  [SKIP] rubric_count=6 already complete (./results_rubric_experiment/0c65ba0b46894372/6.json)

  [SKIP] rubric_count=7 already complete (./results_rubric_experiment/0c65ba0b46894372/7.json)

  [SKIP] rubric_count=8 already complete (./results_rubric_experiment/0c65ba0b46894372/8.json)


In [9]:
import os, json
from pathlib import Path
from collections import defaultdict
 
RESULTS_DIR   = "./results_rubric_experiment"
RUBRIC_COUNTS = [5, 6, 7, 8, 9, 10, "unspecified"]
 
# ── Collect per-condition accuracy from every annotator ──
acc_by_n = defaultdict(list)   # rubric_count_str -> [accuracy, ...]
 
for annotator_dir in sorted(Path(RESULTS_DIR).iterdir()):
    if not annotator_dir.is_dir():
        continue
    for rubric_count in RUBRIC_COUNTS:
        result_path = annotator_dir / f"{rubric_count}.json"
        if not result_path.exists():
            continue
        try:
            with open(result_path) as f:
                data = json.load(f)
            acc = data["evaluation_metrics"]["accuracy"]
            if acc is not None:
                acc_by_n[str(rubric_count)].append(acc)
        except Exception as e:
            print(f"[!] Could not read {result_path}: {e}")
 
# ── Print summary ──
print(f"{'N':<14} {'Avg Acc':>8}  {'Std':>7}  {'n annotators':>13}")
print("-" * 46)
for rubric_count in RUBRIC_COUNTS:
    key    = str(rubric_count)
    values = acc_by_n[key]
    if not values:
        print(f"{key:<14} {'N/A':>8}")
        continue
    avg = sum(values) / len(values)
    std = (sum((v - avg) ** 2 for v in values) / len(values)) ** 0.5
    print(f"{key:<14} {avg*100:>7.1f}%  {std*100:>6.1f}%  {len(values):>13}")
 

N               Avg Acc      Std   n annotators
----------------------------------------------
5                 63.0%     9.7%             20
6                 61.5%    10.5%             20
7                 61.3%     8.8%             20
8                 56.7%    12.7%             20
9                 59.8%     9.0%             20
10                62.0%    10.8%             20
unspecified       59.8%    12.6%             20
